Suite à notre analyse préliminaire des données, nous avons déterminé les entrées et les sorties de notre réseaux de neurones pour prédire le genre d'un film à partir de son affiche de film.\
Dans un premier temps, nous allons mettre en place un classifieur d'image prenant en entrée Poster_Url et renvoyant Genre dans le jeu de données movies.csv\
Voici un schéma de l'architecture envisagée. 

Étant donné le nombre d'affiche par Genre, ChaGPT propose d'utiliser un CNN relativement compact. (justification)\
ChatGPT mentionne qu'avec 9328 images, un modèle comme :\

ResNet-152 ;\
EfficientNet-B7 ;\
Vision Transformer complet ;

risque fortement de sur-apprendre. Et donc, le modèle mémorise les affiches au lieu d'apprendre les caractéristiques des genres.

In [65]:
#On importe les données
import csv
from sklearn.preprocessing import MultiLabelBinarizer

with open("movies.csv", "r") as f:
    data = csv.reader(f)
    rows = list(data)

## On nettoie les données
#On résume les lignes manquantes en une seule ligne
#['Release_Date', 'Title', 'Overview', 'Popularity', 'Vote_Count', 'Vote_Average', 'Original_Language', 'Genre', 'Poster_Url']
rows.append(['2013-10-20', 'Pixie Hollow Bake Off', "Tink challenges Gelata to see who can bake the best cake for the queen's party.  Plus 10 Disney Fairies Mini-Shorts", '61.328', '35', '7.1', 'en', 'Animation', 'https://image.tmdb.org/t/p/original/6iXYe7AkQ1QIfMFuvXsSCT2zF7s.jpg'])
#On supprime les lignes 1106 à 1116 car elles contiennent des données corrompues (erreur de formatage)
del rows[1106:1117]

In [66]:
# On sépare l'en-tête des données et le tri des Genres
header = rows[0]
movies = rows[1:]

# Liste des genres pour chaque film
genres = [
    [g.strip() for g in movie[7].split(",")]
    for movie in movies
]

# Encodage multi-label pour prendre en compte les films avec plusieurs genres
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(genres)

On utilise des multi-label car un film peut appartenir à plusieurs Genres.

On fait en sorte que la variable movies contienne les images

In [67]:
image_paths = [
    f"posters/{i}.jpg"
    for i in range(len(movies))
]

ChatGPT propose la répartition suivante des données\
Train       70%  ≈ 6500 images\
Validation  15%  ≈ 1400 images\
Test        15%  ≈ 1400 images

In [68]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# Premier découpage : 70 % train, 30 % reste
msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    msss.split(image_paths, Y)
)

X_train = [image_paths[i] for i in train_idx]
Y_train = Y[train_idx]

X_temp = [image_paths[i] for i in temp_idx]
Y_temp = Y[temp_idx]

In [79]:
#Vérification
X_train[0]

'posters/0.jpg'

In [80]:
#Vérification
Y_train[0]

array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0])

In [81]:
msss2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.5,
    random_state=42
)

val_idx, test_idx = next(
    msss2.split(X_temp, Y_temp)
)

X_val = [X_temp[i] for i in val_idx]
Y_val = Y_temp[val_idx]

X_test = [X_temp[i] for i in test_idx]
Y_test = Y_temp[test_idx]

In [88]:
#Vérifications
X_test[0]
Y_test[0]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0])

In [96]:
#Vérification de la répartition des données 
print("Train :", len(X_train))
print("Validation :", len(X_val))
print("Test :", len(X_test))

Train : 6890
Validation : 1491
Test : 1446


In [97]:
import numpy as np

#Pour la réutilisation des indices de train, validation et test, on les sauvegarde dans des fichiers .npy
np.save("train_idx.npy", train_idx)
np.save("val_idx.npy", val_idx)
np.save("test_idx.npy", test_idx)

À ce stade les données sont reparties selon les données d'entraînement, de validation et de tests.

Transformons le simages pour qu'elles soient au format attendu par Keras

Afin d'augmenter artificiellement le jeu de données, ChatGPT suggère des retournements, des rotations, des zooms et des mises à l'échelle aléatoires et de faible amplitude.

In [98]:
from tensorflow import keras
from tensorflow.keras import layers

augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.2)
])

On créé la fonction de chargement des images

In [99]:
import tensorflow as tf

IMG_SIZE = (224, 224)

def load_image(path, label):

    # Lire l'image
    image = tf.io.read_file(path)

    # Transformer JPEG → tenseur RGB
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )

    # Redimensionnement
    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    # Normalisation entre 0 et 1
    image = image / 255.0

    # Label en float32 pour BCE
    label = tf.cast(
        label,
        tf.float32
    )

    return image, label

On créé des datasets Tensorflow

In [100]:
BATCH_SIZE = 32

train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, Y_train)
)

train_dataset = (
    train_dataset
    .shuffle(1000)
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [101]:
#Validation
val_dataset = tf.data.Dataset.from_tensor_slices(
    (X_val, Y_val)
)

val_dataset = (
    val_dataset
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


In [102]:
test_dataset = tf.data.Dataset.from_tensor_slices(
    (X_test, Y_test)
)

test_dataset = (
    test_dataset
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [104]:
#Test de l'entraînement
for images, labels in train_dataset.take(1):

    print(images.shape)
    print(labels.shape)

(32, 224, 224, 3)
(32, 19)


2026-06-30 18:31:25.612101: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Voici l'architecure conseillée par ChaGPT, "un CNN relativement compact".\
Image 224×224\
      |\
Conv2D 32 filtres\
      |\
MaxPooling\
      |\
Conv2D 64 filtres\
      |\
MaxPooling\
      |\
Conv2D 128 filtres\
      |\
Global Average Pooling\
      |\
Dense 256\
      |\
Dropout\
      |\
Dense(nombre_de_genres)\
      |\
Sigmoid

In [103]:
from tensorflow.keras import layers, models

nb_genres = 19

model = models.Sequential([
    augmentation,#utile pour augmenter les données d'entrainement, recommandé par ChatGPT
    layers.Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=(224,224,3)
    ),
    layers.MaxPooling2D(),

    layers.Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    layers.MaxPooling2D(),

    layers.Conv2D(
        128,
        (3,3),
        activation="relu"
    ),

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(nb_genres, activation="sigmoid")
])

L'AUC multi-label est souvent plus informative que la simple accuracy.

In [105]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            threshold=0.5
        ),
        tf.keras.metrics.AUC(
            multi_label=True
        )
    ]
)

In [107]:
#Début de l'entraînement
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30
)

Epoch 1/30
196/216 ━━━━━━━━━━━━━━━━━━━━ 7s 384ms/step - auc: 0.5071 - binary_accuracy: 0.8245 - loss: 0.4347

2026-06-30 18:34:36.978912: W tensorflow/core/framework/op_kernel.cc:1855] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: posters/9475.jpg; No such file or directory


199/216 ━━━━━━━━━━━━━━━━━━━━ 6s 384ms/step - auc: 0.5075 - binary_accuracy: 0.8249 - loss: 0.4339

2026-06-30 18:34:38.153111: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: NOT_FOUND: posters/9475.jpg; No such file or directory
	 [[{{node ReadFile}}]]
	 [[IteratorGetNext]]


NotFoundError: Graph execution error:

Detected at node ReadFile defined at (most recent call last):
<stack traces unavailable>
posters/9475.jpg; No such file or directory
	 [[{{node ReadFile}}]]
	 [[IteratorGetNext]] [Op:__inference_multi_step_on_iterator_3987]